In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = [f.parse() for f in files]

len(documents)

72

In [2]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [3]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [4]:
from openai import OpenAI

client = OpenAI()

In [5]:
class RAG:
    def __init__(self, index):
        self.index = index

    def search(self, query):
        return self.index.search(query, num_results=5)

    def build_context(self, results):
        return "\n\n".join(
            f"{r['filename']}\n{r['content']}"
            for r in results
        )

    def llm(self, prompt):
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response

    def ask(self, question):
        results = self.search(question)
        context = self.build_context(results)

        prompt = f"""
Use the context below to answer the question.

CONTEXT:
{context}

QUESTION:
{question}
"""

        response = self.llm(prompt)

        return {
            "answer": response.choices[0].message.content,
            "usage": response.usage
        }

In [7]:
rag = RAG(index)

result = rag.ask(
    "How does the agentic loop keep calling the model until it stops?"
)

print(result["usage"])

CompletionUsage(completion_tokens=138, prompt_tokens=7069, total_tokens=7207, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=6912))


In [8]:
print(result["usage"].completion_tokens)
print(result["usage"].prompt_tokens)



138
7069


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [10]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [12]:
chunk_rag = RAG(chunk_index)

chunk_rag.ask(
    "How does the agentic loop keep calling the model until it stops?"
)["usage"].prompt_tokens

2252

In [14]:
question = "How does the agentic loop keep calling the model until it stops?"

full_result = rag.ask(question)
chunk_result = chunk_rag.ask(question)

full_tokens = full_result["usage"].prompt_tokens
chunk_tokens = chunk_result["usage"].prompt_tokens

diff = full_tokens - chunk_tokens

full_tokens, chunk_tokens, diff

(7069, 2252, 4817)

In [15]:
def search_tool(query: str) -> str:
    """Search course lessons using chunk index."""
    results = chunk_index.search(query, num_results=5)

    return "\n\n".join(
        f"{r['filename']}\n{r['content']}"
        for r in results
    )

In [21]:
from toyaikit.tools import Tools

tools = Tools()

search_calls = []

def search(query: str) -> str:
    """
    Search course lessons using the chunk index.
    """
    global search_calls

    search_calls.append(query)

    results = chunk_index.search(query, num_results=5)

    return "\n\n".join([
        f"FILE: {r['filename']}\n{r['content']}"
        for r in results
    ])

tools.add_tool(search)

In [22]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

openai_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI()
)

In [23]:
developer_prompt = """
You're a course teaching assistant.

Answer the student's question using the search tool.

Make multiple searches with different keywords before answering.
"""

In [24]:
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=openai_client
)

In [29]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)

In [31]:
print(len(search_calls))

3
